# Dataset Factory: Pythia-410M / 1B on Colab (Module 10d)

Continued pretraining of a released Pythia checkpoint on a corpus packed locally with
`python -m dftrain corpus`, then evaluation (WikiText / medical perplexity, PubMedQA) and
QLoRA on PubMedQA. Everything is written to Google Drive, and pretraining saves a checkpoint
every `SAVE_EVERY` steps and stops at a step limit per session, so a dead or time-limited
session loses at most `SAVE_EVERY` steps: just run the notebook again.

**Before you start (on your PC):**
1. Zip the `training` folder **without** `data/runs` and `data/experiments` (only `dftrain/`,
   `pyproject.toml`, `data/eval/`, and the corpora you want under `data/corpora/`).
2. Upload it to Google Drive as `MyDrive/DatasetFactory/training.zip`.

**GPU memory (full-parameter training, AdamW, fp16):** Pythia-410M fits a 16 GB T4 with
`MICRO_BATCH = 2` and gradient checkpointing. Pythia-1B needs ~16 GB for weights + Adam
states alone, so use an A100 (40 GB) or L4 (24 GB, with gradient checkpointing and
`MICRO_BATCH = 1`). QLoRA (4-bit) of either size fits any Colab GPU.
Precision is fp16 (bf16 measured harmful for Pythia).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/DatasetFactory')
os.makedirs(WORK, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Unpack the code + eval sets + corpora next to the notebook's Drive folder (once).
if not (WORK / 'training' / 'dftrain').exists():
    !cd "{WORK}" && unzip -q training.zip -d training_unzip && \
        (test -d training_unzip/training && mv training_unzip/training training || mv training_unzip training)
os.chdir(WORK / 'training')
!pip -q install "transformers>=4.45" accelerate peft bitsandbytes tokenizers huggingface_hub datasets
!ls data/eval data/corpora

## Settings

`CORPORA` are folders under `data/corpora` (built locally with `dftrain corpus ... --budget 200M`).
`STEPS_PER_SESSION` caps the steps in one run of the next cell; raise it on a paid GPU.
Local defaults are kept (peak lr 6e-5, warmup 5%, linear decay);
`GLOBAL_BATCH = 256` sequences x 1024 tokens is the paper's batch (200M tokens -> 762 steps).

In [ ]:
MODEL = 'EleutherAI/pythia-410m'     # or 'EleutherAI/pythia-1b'
CORPORA = ['curated-200m', 'fineweb-200m', 'random-200m']
SEEDS = [0]
GLOBAL_BATCH = 256
MICRO_BATCH = 2                      # 410M on a T4: 2 with checkpointing; A100: 8
GRAD_CKPT = True
LR = 6e-5
SAVE_EVERY = 25                      # steps between checkpoints on Drive
STEPS_PER_SESSION = 0                # 0 = no limit; e.g. 200 to stop cleanly before a timeout
size = MODEL.split('-')[-1]

In [ ]:
import json, time, pathlib
from dftrain.pretrain import TrainConfig, train

def pretrain(corpus, seed):
    out = pathlib.Path(f'data/runs/{corpus}-{size}-s{seed}')
    if (out / 'summary.json').exists():
        print('done:', out); return out
    resume = (out / 'checkpoint.pt').exists()
    step0 = 0
    if resume:
        import torch
        step0 = torch.load(out / 'checkpoint.pt', map_location='cpu', weights_only=False)['step']
    cfg = TrainConfig(corpus=f'data/corpora/{corpus}', out=str(out), model=MODEL, seed=seed,
                      global_batch=GLOBAL_BATCH, micro_batch=MICRO_BATCH, lr=LR,
                      save_every=SAVE_EVERY, gradient_checkpointing=GRAD_CKPT,
                      stop_at=step0 + STEPS_PER_SESSION if STEPS_PER_SESSION else 0)
    r = train(cfg, resume=resume)
    print(r)
    return out

for seed in SEEDS:
    for c in CORPORA:
        pretrain(c, seed)

## Evaluation

Same frozen evaluation sets as locally (`data/eval`, check the sha256 in each result file).
Report macro-F1 and the prediction counts next to accuracy: always answering "yes" gives 55.2%.

In [ ]:
from dftrain.evaluate import evaluate, compare
EVAL = pathlib.Path('data/eval')
res_dir = pathlib.Path(f'data/results/colab-{size}'); res_dir.mkdir(parents=True, exist_ok=True)

def ev(model_path, name, **kw):
    out = res_dir / f'{name}.json'
    if out.exists():
        return json.loads(out.read_text())
    return evaluate(model_path, EVAL, out, shots=(0, 3), seed=0, **kw)

results = {'base': ev(MODEL, 'base')}
for seed in SEEDS:
    for c in CORPORA:
        run = pathlib.Path(f'data/runs/{c}-{size}-s{seed}')
        if (run / 'model').exists():
            results[f'{c}-s{seed}'] = ev(str(run / 'model'), f'{c}-s{seed}')

for k, r in results.items():
    q = r['pubmedqa']['0shot']['raw']
    print(f"{k:24s} wikitext {r['wikitext']['perplexity']:7.2f}  medical {r['medical']['perplexity']:7.2f}"
          f"  pubmedqa0 {q['accuracy']:.3f} F1 {q['macro_f1']:.3f} pred {q['predicted']}")

In [ ]:
# Paired bootstrap of each corpus vs the first one (per seed)
ref = CORPORA[0]
for seed in SEEDS:
    for c in CORPORA[1:]:
        a, b = results.get(f'{ref}-s{seed}'), results.get(f'{c}-s{seed}')
        if a and b:
            print(f'{c} vs {ref}, seed {seed}:', json.dumps(compare(a, b), indent=1))

## QLoRA on PubMedQA

Paper Table 4 settings (r 16, alpha 32, dropout 0.05, `query_key_value`, lr 2e-4, 3 epochs,
batch 4 x 4), 4-bit NF4 base, trained on the 500 `pubmedqa_train` items, evaluated 0-shot on the
500 test items. Locally (RTX 4060) Pythia-160M only learned the label prior (always "yes");
410M learned real signal.

In [ ]:
from dftrain.qlora import QLoRAConfig, train as qtrain

def qlora(model_path, name, seed):
    run = pathlib.Path(f'data/runs/qlora-{name}-s{seed}')
    if not (run / 'summary.json').exists():
        qtrain(QLoRAConfig(model=model_path, out=str(run), seed=seed, gradient_checkpointing=True))
    out = res_dir / f'qlora-{name}-s{seed}.json'
    if not out.exists():
        evaluate(str(run), EVAL, out, tasks=['pubmedqa'], shots=(0,), seed=0)
    return json.loads(out.read_text())

for seed in SEEDS:
    for name, path in [('base', MODEL)] + [(f'{c}-{size}', f'data/runs/{c}-{size}-s{seed}/model')
                                           for c in CORPORA]:
        if path == MODEL or pathlib.Path(path).exists():
            q = qlora(path, name, seed)['pubmedqa']['0shot']['raw']
            print(f"{name:24s} seed {seed}: acc {q['accuracy']:.3f} F1 {q['macro_f1']:.3f} pred {q['predicted']}")

## Results back to the PC

Everything is on Drive under `DatasetFactory/training/data/results/colab-<size>` and
`data/runs/...` (the models are large; the result JSON files are what the experiment summary needs).